# Notebook 11: The Scikit-Learn API — fit, transform, predict, pipelines

**Difficulty:** ⭐⭐⭐ (Intermediate) | **Estimated Time:** 3 hours

**Week 3 — Statistics for ML & Python Data Stack**

---

## What you will learn

- The three-method API that unifies ALL of scikit-learn: `.fit()`, `.transform()`, `.predict()`
- Preprocessing: scaling, imputation, encoding categorical variables
- Training and evaluating regression models (LinearRegression, DecisionTree, KNeighbors)
- Building clean, leak-free Pipelines and ColumnTransformers
- Cross-validation: comparing models reliably

**Running theme:** Predicting house prices from size, bedrooms, neighborhood, and condition.

## Why does this matter?

Scikit-learn is the backbone of applied machine learning in Python. Nearly every real-world ML project — whether you work at a startup, a bank, or a research lab — begins with scikit-learn.

But scikit-learn's real superpower is not any single algorithm. It's the **unified API**: every single model and preprocessor in the library obeys the same three-method contract. Once you understand `.fit()`, `.transform()`, and `.predict()`, you can use **any** of sklearn's 100+ estimators without re-learning the interface.

This consistency unlocks:
- **Swappable models:** Change `LinearRegression` to `RandomForestRegressor` by editing one line.
- **Pipelines:** Chain preprocessing and modeling into a single, leak-proof object.
- **Model selection tools** like `cross_val_score` that work with *any* estimator automatically.

Understanding this notebook is the foundation for everything that follows in this course.

## Real-World Analogy: The Kitchen Appliance Store

Imagine you walk into a kitchen appliance store. There are blenders, coffee makers, toasters, and food processors — dozens of different machines. But here is the thing: **they all work the same way**.

1. **Plug it in** — connect it to power (`.fit()`: give it data to learn from)
2. **Set up** — choose your settings, load the ingredients (configure the estimator)
3. **Press start** — process the food (`.transform()` or `.predict()`: apply to new data)

You don't need a separate instruction manual for each appliance. The interface is consistent.

Scikit-learn works exactly the same way:
- A `StandardScaler` and a `LinearRegression` look very different internally, but you use them identically.
- You `.fit()` on training data, then `.transform()` or `.predict()` on new data.
- Want to try a different 'appliance'? Swap the class name — the rest of your code stays the same.

```
Kitchen appliances        sklearn estimators
------------------        ------------------
Plug in               -->  .fit(X_train, y_train)
Set settings          -->  constructor arguments (n_estimators=100)
Put food in           -->  .transform(X_test) or .predict(X_test)
Get output            -->  scaled array / predicted prices
Swap blender for mixer --> swap LinearRegression for RandomForest
```

## Section 0: Imports and Synthetic Dataset

Before diving into the API, let's build a realistic synthetic house price dataset. We'll use it throughout all sections.

**The dataset contains:**
- `size_sqft` — house size in square feet (300–3500)
- `bedrooms` — number of bedrooms (1–6)
- `bathrooms` — number of bathrooms (1–4)
- `age_years` — age of the house in years (0–80)
- `neighborhood` — categorical: 'downtown', 'suburbs', 'rural'
- `condition` — ordinal: 'poor', 'fair', 'good', 'excellent'
- `price` — target variable: house price in USD

We'll also introduce some `NaN` (missing) values to practice imputation.

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn: preprocessing ─────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler,      # subtract mean, divide by std
    MinMaxScaler,        # scale to [0, 1]
    LabelEncoder,        # ordinal categories → integers
    OneHotEncoder        # nominal categories → binary columns
)
from sklearn.impute import SimpleImputer          # fill NaN values
from sklearn.compose import ColumnTransformer     # different transforms per column
from sklearn.pipeline import Pipeline             # chain steps end-to-end

# ── Scikit-learn: models ─────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

# ── Scikit-learn: model selection & evaluation ───────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Reproducibility ──────────────────────────────────────────────────────────
np.random.seed(42)           # fix random seed so results are reproducible

# ── Plot style ───────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All imports successful!')

In [ ]:
# ── Build synthetic house price dataset ─────────────────────────────────────
N = 300   # 300 houses in our dataset

# Numeric features
size_sqft  = np.random.randint(300, 3500, N).astype(float)
bedrooms   = np.random.randint(1, 7, N).astype(float)      # 1–6 bedrooms
bathrooms  = np.random.randint(1, 5, N).astype(float)      # 1–4 bathrooms
age_years  = np.random.randint(0, 81, N).astype(float)     # 0–80 years old

# Categorical feature: neighborhood type
neighborhoods = np.random.choice(['downtown', 'suburbs', 'rural'], N,
                                  p=[0.3, 0.5, 0.2])        # 30/50/20 split

# Ordinal feature: house condition
conditions = np.random.choice(['poor', 'fair', 'good', 'excellent'], N,
                               p=[0.1, 0.25, 0.45, 0.20])

# Condition encoding for price formula (ordinal numeric values)
condition_map = {'poor': 0, 'fair': 1, 'good': 2, 'excellent': 3}
condition_num = np.array([condition_map[c] for c in conditions])

# Neighborhood encoding for price formula
nbhd_map = {'rural': 0, 'suburbs': 1, 'downtown': 2}
nbhd_num = np.array([nbhd_map[n] for n in neighborhoods])

# Price formula: realistic relationship with noise
# Base price driven by size, boosted by neighborhood & condition, penalised by age
price = (
    80 * size_sqft                       # $80 per sqft base
    + 10_000 * bedrooms                  # each bedroom adds value
    + 8_000 * bathrooms                  # each bathroom adds value
    - 1_500 * age_years                  # older houses worth less
    + 25_000 * nbhd_num                  # downtown premium
    + 15_000 * condition_num             # better condition = higher price
    + np.random.normal(0, 20_000, N)     # realistic noise / unexplained variance
).clip(50_000, 1_200_000)               # keep prices in a realistic range

# Assemble into a DataFrame
df = pd.DataFrame({
    'size_sqft':    size_sqft,
    'bedrooms':     bedrooms,
    'bathrooms':    bathrooms,
    'age_years':    age_years,
    'neighborhood': neighborhoods,
    'condition':    conditions,
    'price':        price
})

# ── Introduce realistic missing values ──────────────────────────────────────
# About 5% missing in size_sqft (inspector didn't record it)
missing_size = np.random.choice(N, size=15, replace=False)
df.loc[missing_size, 'size_sqft'] = np.nan

# About 3% missing in age_years
missing_age = np.random.choice(N, size=9, replace=False)
df.loc[missing_age, 'age_years'] = np.nan

print(f'Dataset shape: {df.shape}')
print(f'\nMissing values per column:')
print(df.isnull().sum())
print(f'\nFirst 5 rows:')
df.head()

## Section 1: Preprocessing — The Three-Method API in Action

### Plain English: What is preprocessing?

Raw data is messy. Before feeding it to a model, we need to:

1. **Fill in missing values** (imputation) — a model can't handle `NaN`
2. **Scale numeric features** — a feature measured in thousands shouldn't dominate one measured in units
3. **Encode categories** — 'downtown'/'suburbs'/'rural' needs to become numbers

All of these preprocessing steps in sklearn follow the **exact same API**:
- `.fit(X)` — compute statistics from the training data (the mean, the min/max, the unique categories)
- `.transform(X)` — apply the transformation to data
- `.fit_transform(X)` — do both in one step (for training data only!)

### The Golden Rule of Preprocessing

**ALWAYS** fit your preprocessors on training data only. Never on the full dataset. Why?

If you compute the mean on the full dataset (including test data), your model has "peeked" at the test set. This is called **data leakage** — like studying with the actual exam answers. Your results will look better than they really are, but you'll fail in production.

In [ ]:
# ── StandardScaler: subtract mean, divide by std ────────────────────────────
#
# Analogy: imagine comparing marathon times in seconds vs hours.
# Seconds range: 7200–14400. Hours range: 2.0–4.0.
# Without scaling, models give more weight to "seconds" just because
# the numbers are bigger — even though the INFORMATION is the same.
# StandardScaler puts everything on the same playing field.

# Use only the two numeric columns for this demo
demo_data = df[['size_sqft', 'age_years']].dropna()   # remove NaNs first

# Step 1: Create the scaler object (like unboxing the kitchen appliance)
scaler = StandardScaler()

# Step 2: .fit() — compute mean and std FROM the data
# The scaler learns:  mean_size, std_size, mean_age, std_age
scaler.fit(demo_data)

print('After .fit(), the scaler has learned:')
print(f'  Mean of size_sqft : {scaler.mean_[0]:,.1f}')
print(f'  Mean of age_years : {scaler.mean_[1]:.1f}')
print(f'  Std  of size_sqft : {scaler.scale_[0]:,.1f}')
print(f'  Std  of age_years : {scaler.scale_[1]:.1f}')

# Step 3: .transform() — apply scaling: z = (x - mean) / std
scaled_data = scaler.transform(demo_data)

# Step 4: Compare before and after
print('\n--- BEFORE scaling ---')
print(f'  size_sqft: mean={demo_data["size_sqft"].mean():.1f},  std={demo_data["size_sqft"].std():.1f}')
print(f'  age_years: mean={demo_data["age_years"].mean():.1f},  std={demo_data["age_years"].std():.1f}')

print('\n--- AFTER scaling (should be approx mean=0, std=1) ---')
print(f'  size_sqft: mean={scaled_data[:, 0].mean():.6f},  std={scaled_data[:, 0].std():.6f}')
print(f'  age_years: mean={scaled_data[:, 1].mean():.6f},  std={scaled_data[:, 1].std():.6f}')

# Note: fit_transform() does fit + transform in ONE call
# scaled_data = scaler.fit_transform(demo_data)  # equivalent but shorter

In [ ]:
# ── Visualize: scaling effect on distributions ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('StandardScaler: Before vs After Scaling', fontsize=15, fontweight='bold')

# Before: size_sqft
axes[0, 0].hist(demo_data['size_sqft'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('size_sqft — BEFORE scaling', fontsize=12)
axes[0, 0].set_xlabel('Square Feet')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(demo_data['size_sqft'].mean(), color='red', linestyle='--', label=f'Mean={demo_data["size_sqft"].mean():.0f}')
axes[0, 0].legend()

# After: size_sqft (column 0)
axes[0, 1].hist(scaled_data[:, 0], bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('size_sqft — AFTER scaling (z-scores)', fontsize=12)
axes[0, 1].set_xlabel('Standard Deviations from Mean')
axes[0, 1].axvline(0, color='red', linestyle='--', label='Mean=0')
axes[0, 1].legend()

# Before: age_years
axes[1, 0].hist(demo_data['age_years'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('age_years — BEFORE scaling', fontsize=12)
axes[1, 0].set_xlabel('Years')
axes[1, 0].set_ylabel('Count')
axes[1, 0].axvline(demo_data['age_years'].mean(), color='red', linestyle='--', label=f'Mean={demo_data["age_years"].mean():.0f}')
axes[1, 0].legend()

# After: age_years (column 1)
axes[1, 1].hist(scaled_data[:, 1], bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('age_years — AFTER scaling (z-scores)', fontsize=12)
axes[1, 1].set_xlabel('Standard Deviations from Mean')
axes[1, 1].axvline(0, color='red', linestyle='--', label='Mean=0')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print('Key insight: The SHAPE of the distribution is unchanged after scaling.')
print('Only the axis units change — now both features live on the same scale.')

In [ ]:
# ── MinMaxScaler: scale to [0, 1] ────────────────────────────────────────────
# Formula: x_scaled = (x - x_min) / (x_max - x_min)
#
# When to use MinMaxScaler vs StandardScaler?
#   StandardScaler: preferred when data is roughly normal; used with most models
#   MinMaxScaler:   when you NEED values bounded in [0,1] (e.g. neural networks,
#                   image pixel values); sensitive to outliers

mm_scaler = MinMaxScaler()   # scale to range [0, 1] by default

# fit_transform is a shortcut for .fit() then .transform()
mm_scaled = mm_scaler.fit_transform(demo_data)

print('MinMaxScaler results:')
print(f'  size_sqft: min={mm_scaled[:, 0].min():.4f}, max={mm_scaled[:, 0].max():.4f}')
print(f'  age_years: min={mm_scaled[:, 1].min():.4f}, max={mm_scaled[:, 1].max():.4f}')
print('  (Both are now in [0, 1] range)')

# ── SimpleImputer: fill missing (NaN) values ──────────────────────────────
# Analogy: if a form has a blank field, you fill it with a reasonable default.
# 'mean' strategy: replace NaN with the column average
# 'median' strategy: replace NaN with the column median (robust to outliers)
# 'most_frequent': replace NaN with the most common value (good for categoricals)

numeric_cols = df[['size_sqft', 'age_years']]  # these have NaN values

imputer = SimpleImputer(strategy='median')    # median is robust to outliers
imputer.fit(numeric_cols)                     # learns: median of each column

print(f'\nSimpleImputer learned medians:')
print(f'  median size_sqft : {imputer.statistics_[0]:,.1f}')
print(f'  median age_years : {imputer.statistics_[1]:.1f}')

filled = imputer.transform(numeric_cols)       # NaNs replaced with medians

print(f'\nBefore imputation — NaN count: {numeric_cols.isnull().sum().sum()}')
print(f'After  imputation — NaN count: {pd.DataFrame(filled).isnull().sum().sum()}')

In [ ]:
# ── OneHotEncoder: categorical → binary columns ──────────────────────────────
#
# Problem: 'downtown', 'suburbs', 'rural' are strings.
# We CANNOT just encode them as 0, 1, 2 — that implies downtown > suburbs > rural,
# which is wrong. These are NOMINAL (no inherent order).
#
# Solution: One-Hot Encoding — create one binary column per category.
# 'downtown' → [1, 0, 0]
# 'suburbs'  → [0, 1, 0]
# 'rural'    → [0, 0, 1]
#
# The "dummy variable trap": if we know two values, the third is implied.
# drop='first' removes one column to avoid this multicollinearity.

nbhd_column = df[['neighborhood']]    # must be 2D array for sklearn

ohe = OneHotEncoder(drop='first',     # drop first category to avoid trap
                    sparse_output=False)  # return dense array (not sparse matrix)

ohe.fit(nbhd_column)   # learns the unique categories

print('OneHotEncoder learned categories:', ohe.categories_)
print('After drop="first", column names:', ohe.get_feature_names_out())

# Transform: string → binary columns
encoded = ohe.transform(nbhd_column)

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print('\nSample of encoded neighborhoods:')
# Show original alongside encoded
result = pd.concat([df['neighborhood'].reset_index(drop=True), encoded_df], axis=1)
print(result.head(8))

# ── LabelEncoder: ordinal categories → integers ──────────────────────────────
#
# For ORDINAL features (there IS a meaningful order), we CAN use integers.
# poor=0, fair=1, good=2, excellent=3  — this ordering is meaningful.

# LabelEncoder needs a manual mapping for custom ordinal order.
# (LabelEncoder itself uses alphabetical order, which would give wrong results here)
condition_order = {'poor': 0, 'fair': 1, 'good': 2, 'excellent': 3}
df['condition_encoded'] = df['condition'].map(condition_order)

print('\nCondition ordinal encoding sample:')
print(df[['condition', 'condition_encoded']].drop_duplicates().sort_values('condition_encoded'))

## Section 2: Models — The Same API for Everything

### Plain English: Why the unified API is a superpower

Here is a profound fact about scikit-learn: once you learn how to use one model, you know how to use all of them. The API is identical:

```python
model = SomeModel(hyperparameter=value)
model.fit(X_train, y_train)    # learn from training data
predictions = model.predict(X_test)  # apply to new data
score = model.score(X_test, y_test)  # evaluate
```

We'll now see three very different models — Linear Regression, Decision Tree, and K-Nearest Neighbors — all using this identical pattern.

**Quick model descriptions:**
- **LinearRegression**: finds the best straight-line relationship between features and price
- **DecisionTreeRegressor**: learns a series of if/else rules ('if size > 2000 and downtown, price = ...')
- **KNeighborsRegressor**: predicts a house's price as the average price of the K most similar houses

These are completely different algorithms under the hood. But from the outside? Same API.

In [ ]:
# ── Prepare a clean numeric-only feature matrix for model comparison ─────────
# We'll use the encoded/imputed features

# First, manually impute and encode (we'll automate this with Pipelines later)

df_clean = df.copy()

# Fill missing numeric values with median
df_clean['size_sqft'].fillna(df_clean['size_sqft'].median(), inplace=True)
df_clean['age_years'].fillna(df_clean['age_years'].median(), inplace=True)

# Encode neighborhood with one-hot (we already created condition_encoded)
neighborhood_dummies = pd.get_dummies(df_clean['neighborhood'], prefix='nbhd', drop_first=True)

# Assemble final feature matrix
X = pd.concat([
    df_clean[['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'condition_encoded']],
    neighborhood_dummies
], axis=1)

y = df_clean['price']    # target variable

print(f'Feature matrix shape: {X.shape}')
print(f'Feature columns: {list(X.columns)}')
print(f'Target shape: {y.shape}')
print(f'Price range: ${y.min():,.0f} — ${y.max():,.0f}')

# ── Train/test split ──────────────────────────────────────────────────────────
# Why split? If we test on training data, we're checking if the model memorised
# the answers — not whether it learned generalizable patterns.
# Analogy: studying a textbook, then being tested on NEW problems — not the same ones.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing → 60 houses
    random_state=42     # fix seed for reproducibility
)

print(f'\nTrain size: {X_train.shape[0]} houses')
print(f'Test  size: {X_test.shape[0]} houses')

In [ ]:
# ── Scale features (fit on TRAIN only!) ──────────────────────────────────────
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)   # fit AND transform train
X_test_scaled  = scaler_model.transform(X_test)        # ONLY transform test
# ^ Key: we do NOT call fit_transform on the test set.
#   We use the mean/std LEARNED from training data.

# ── Model 1: Linear Regression ───────────────────────────────────────────────
lr = LinearRegression()                    # create the model object
lr.fit(X_train_scaled, y_train)            # learn: find coefficients minimizing MSE
y_pred_lr = lr.predict(X_test_scaled)      # apply to unseen test houses

r2_lr  = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print('── Linear Regression ──────────────────────')
print(f'  R²   = {r2_lr:.4f}  (fraction of price variance explained)')
print(f'  MAE  = ${mae_lr:,.0f}  (avg absolute prediction error)')
print(f'  RMSE = ${rmse_lr:,.0f}  (root mean squared error)')

# Inspect learned coefficients — tells you which features matter most
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr.coef_
}).sort_values('coefficient', ascending=False)

print(f'\n  Intercept: ${lr.intercept_:,.0f}')
print('  Coefficients (scaled features):')
print(coef_df.to_string(index=False))

In [ ]:
# ── Model 2: Decision Tree Regressor ─────────────────────────────────────────
# Same API! Only the class name changes.
dt = DecisionTreeRegressor(
    max_depth=5,        # limit tree depth to avoid overfitting
    random_state=42
)
dt.fit(X_train_scaled, y_train)             # learn: build decision tree
y_pred_dt = dt.predict(X_test_scaled)       # predict: traverse tree for each house

r2_dt   = r2_score(y_test, y_pred_dt)
mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print('── Decision Tree Regressor ────────────────')
print(f'  R²   = {r2_dt:.4f}')
print(f'  MAE  = ${mae_dt:,.0f}')
print(f'  RMSE = ${rmse_dt:,.0f}')

# ── Model 3: K-Nearest Neighbors Regressor ───────────────────────────────────
# Predicts price as the average of the K most similar houses.
# k=5: for each new house, find the 5 most similar training houses,
# and average their prices.
knn = KNeighborsRegressor(
    n_neighbors=5,      # use 5 nearest neighbors
    weights='distance'  # closer neighbors have more influence
)
knn.fit(X_train_scaled, y_train)            # "learn": just store training data
y_pred_knn = knn.predict(X_test_scaled)     # find 5 nearest, average their prices

r2_knn   = r2_score(y_test, y_pred_knn)
mae_knn  = mean_absolute_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print('\n── K-Nearest Neighbors Regressor ──────────')
print(f'  R²   = {r2_knn:.4f}')
print(f'  MAE  = ${mae_knn:,.0f}')
print(f'  RMSE = ${rmse_knn:,.0f}')

print('\n──────────────────────────────────────────────────────')
print('Note: All three models used IDENTICAL code structure!')
print('Only the class name and hyperparameters differed.')

In [ ]:
# ── Visualize: Actual vs Predicted prices for all three models ───────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Actual vs Predicted House Prices — Three Models', fontsize=14, fontweight='bold')

models_results = [
    ('Linear Regression', y_pred_lr, r2_lr, 'steelblue'),
    ('Decision Tree',     y_pred_dt, r2_dt, 'coral'),
    ('KNN',               y_pred_knn, r2_knn, 'mediumseagreen'),
]

price_range = [y_test.min(), y_test.max()]   # for the perfect prediction line

for ax, (name, y_pred, r2, color) in zip(axes, models_results):
    ax.scatter(y_test, y_pred, alpha=0.5, color=color, edgecolor='white', s=40)

    # Perfect prediction line: if model were perfect, all points would lie on this
    ax.plot(price_range, price_range, 'k--', linewidth=2, label='Perfect prediction')

    ax.set_title(f'{name}\nR² = {r2:.3f}', fontsize=12)
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.legend(fontsize=9)

    # Format axis tick labels as $K
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Points close to the dashed line = good predictions.')
print('  R² closer to 1.0 = better model.')
print('  Spread around the line = prediction error.')

## Section 3: Model Evaluation in Depth

### Plain English: What do these metrics mean?

We've been throwing metrics around — let's understand what they actually tell us.

**R² (R-squared / Coefficient of Determination)**
- Ranges from 0 to 1 (can be negative for very bad models)
- Interpretation: "What fraction of the variance in house prices is explained by our features?"
- R² = 0.80 means: our model explains 80% of why prices vary between houses
- The remaining 20% is due to factors our model doesn't capture (luck of the market, specific renovations, etc.)

**MAE (Mean Absolute Error)**
- Average of |actual - predicted| for all houses
- Easy to interpret: if MAE = \$15,000, you're off by \$15K on average
- Robust to outliers — a single terrible prediction doesn't dominate

**RMSE (Root Mean Squared Error)**
- Square root of the average squared errors
- Penalises large errors more than MAE — one \$100K error hurts much more than ten \$10K errors
- Good when large errors are particularly bad (e.g. a bank giving a wrong \$200K mortgage quote)

### The Train/Test Split explained

Analogy: a student studying for an exam. If the teacher tests them on the exact problems they practised (train = test), even a student who just memorised the answers will score 100%. We need NEW problems to see if they truly understood the material.

```
Full dataset (300 houses)
        │
        ├── Training set (240 houses, 80%) → model LEARNS from these
        │
        └── Test set (60 houses, 20%) → model is EVALUATED on these
                                         (model has NEVER seen these)
```

In [ ]:
# ── Comprehensive evaluation: training vs test performance ───────────────────
# This reveals overfitting: a model that memorised training data will score
# high on train but poorly on test.

print('='*60)
print('MODEL EVALUATION: Training vs Test Performance')
print('='*60)

for name, model, y_pred_te in [
    ('Linear Regression', lr, y_pred_lr),
    ('Decision Tree (depth=5)', dt, y_pred_dt),
    ('KNN (k=5)', knn, y_pred_knn),
]:
    # Evaluate on training data too (to check for overfitting)
    y_pred_tr = model.predict(X_train_scaled)

    r2_train = r2_score(y_train, y_pred_tr)
    r2_test  = r2_score(y_test, y_pred_te)

    mae_train = mean_absolute_error(y_train, y_pred_tr)
    mae_test  = mean_absolute_error(y_test, y_pred_te)

    print(f'\n{name}')
    print(f'  R²  — Train: {r2_train:.4f}  |  Test: {r2_test:.4f}  |  Gap: {r2_train - r2_test:.4f}')
    print(f'  MAE — Train: ${mae_train:,.0f}  |  Test: ${mae_test:,.0f}')

    # Diagnose overfit/underfit
    gap = r2_train - r2_test
    if gap > 0.15:
        print(f'  WARNING: Large train/test R² gap ({gap:.3f}) → likely OVERFITTING')
    elif r2_test < 0.5:
        print(f'  NOTE: Low test R² → model may be UNDERFITTING')
    else:
        print(f'  Status: Reasonable generalisation')

print('\n' + '='*60)

## Section 4: Pipelines — The Clean Way to Chain Steps

### The Problem Without Pipelines

So far, our workflow has been:

```python
# 1. Impute missing values
imputer = SimpleImputer(); imputer.fit(X_train); X_train = imputer.transform(X_train)
# 2. Scale features
scaler = StandardScaler(); scaler.fit(X_train); X_train = scaler.transform(X_train)
# 3. Fit model
lr = LinearRegression(); lr.fit(X_train, y_train)
# ... and then remember to apply imputer and scaler to X_test before predicting!
```

This is error-prone. It's easy to forget to apply the imputer to new data, or accidentally fit the scaler on test data.

### Pipelines: The Assembly Line Analogy

Think of a car assembly line. Each station has one job:
- **Station 1**: Weld the chassis
- **Station 2**: Paint
- **Station 3**: Install engine

The car moves through each station in order. You don't manually carry it from station to station — the belt does it automatically.

A sklearn `Pipeline` is exactly this: a sequence of processing steps where data flows automatically from one step to the next.

```
Raw house data
     │
     ▼  Step 1: SimpleImputer    (fill missing values)
     │
     ▼  Step 2: StandardScaler   (normalise features)
     │
     ▼  Step 3: LinearRegression (make predictions)
     │
     ▼  Predicted house price
```

**Key benefit:** call `pipeline.fit(X_train, y_train)` ONCE and it fits all steps in sequence, correctly.

In [ ]:
# ── Manual workflow (the messy way) vs Pipeline (the clean way) ───────────────
# We'll demonstrate BOTH on the SAME data, then show the results are identical.

# --- MANUAL WORKFLOW (error-prone) -------------------------------------------
print('MANUAL WORKFLOW:')
print('-' * 40)

# Using only numeric features for this comparison
numeric_features = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'condition_encoded']
X_num = df_clean[numeric_features]    # numeric-only feature set

X_num_train, X_num_test, y_num_train, y_num_test = train_test_split(
    X_num, y, test_size=0.2, random_state=42
)

# Step 1: impute
manual_imputer = SimpleImputer(strategy='median')
X_num_train_imp = manual_imputer.fit_transform(X_num_train)   # fit+transform train
X_num_test_imp  = manual_imputer.transform(X_num_test)        # only transform test

# Step 2: scale
manual_scaler = StandardScaler()
X_num_train_sc = manual_scaler.fit_transform(X_num_train_imp) # fit+transform train
X_num_test_sc  = manual_scaler.transform(X_num_test_imp)      # only transform test

# Step 3: model
manual_lr = LinearRegression()
manual_lr.fit(X_num_train_sc, y_num_train)
manual_preds = manual_lr.predict(X_num_test_sc)
manual_r2 = r2_score(y_num_test, manual_preds)

print(f'  Lines of code for train: ~6')
print(f'  Lines of code for test:  ~3')
print(f'  R² = {manual_r2:.4f}')
print(f'  (Risk: forgetting to apply imputer/scaler to test data)')

# --- PIPELINE WORKFLOW (clean, safe) -----------------------------------------
print('\nPIPELINE WORKFLOW:')
print('-' * 40)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # step name: 'imputer'
    ('scaler',  StandardScaler()),                  # step name: 'scaler'
    ('model',   LinearRegression())                 # step name: 'model'
])

# ONE call: fits imputer, scaler, model — all on training data
pipe.fit(X_num_train, y_num_train)

# ONE call: applies imputer → scaler → model.predict  automatically
pipe_preds = pipe.predict(X_num_test)
pipe_r2 = r2_score(y_num_test, pipe_preds)

print(f'  Lines of code for train: 1 (pipe.fit)')
print(f'  Lines of code for test:  1 (pipe.predict)')
print(f'  R² = {pipe_r2:.4f}')
print(f'  (No risk: pipeline handles transformations automatically)')

print(f'\nResults identical? {abs(manual_r2 - pipe_r2) < 1e-10}')

In [ ]:
# ── ColumnTransformer: different preprocessing for different column types ─────
#
# Real datasets have a mix of numeric and categorical columns.
# We need:
#   numeric columns  → impute with median, then StandardScaler
#   categorical cols → impute with most_frequent, then OneHotEncoder
#
# ColumnTransformer lets us define these different transformations
# and applies each to the appropriate columns.

# Define which columns are which type
numeric_cols    = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'condition_encoded']
categorical_cols = ['neighborhood']

# Build sub-pipelines for each column type
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # fill NaN with median
    ('scaler',  StandardScaler())                    # standardise
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # fill NaN with mode
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))  # one-hot encode
])

# ColumnTransformer: apply the right transformer to each column group
preprocessor = ColumnTransformer([
    ('num',  numeric_transformer,     numeric_cols),      # ('name', transformer, columns)
    ('cat',  categorical_transformer, categorical_cols)
])

# Full pipeline: preprocessing + model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        LinearRegression())
])

# Use raw df (with NaN values and categorical strings) — pipeline handles everything!
X_raw = df[numeric_cols + categorical_cols]
y_raw = df['price']

X_raw_train, X_raw_test, y_raw_train, y_raw_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# ONE LINE to train the entire preprocessing + model pipeline
full_pipeline.fit(X_raw_train, y_raw_train)

# ONE LINE to predict (raw data with NaN and strings goes in, predictions come out)
full_preds = full_pipeline.predict(X_raw_test)
full_r2 = r2_score(y_raw_test, full_preds)
full_mae = mean_absolute_error(y_raw_test, full_preds)

print('Full Pipeline (ColumnTransformer + LinearRegression):')
print(f'  Input:  Raw DataFrame with NaN values and categorical strings')
print(f'  Output: Predicted house prices')
print(f'  R²  = {full_r2:.4f}')
print(f'  MAE = ${full_mae:,.0f}')
print(f'\nPipeline steps:')
for step_name, step_obj in full_pipeline.steps:
    print(f'  {step_name}: {type(step_obj).__name__}')

In [ ]:
# ── Visualize: Pipeline flow diagram ─────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Full Pipeline Architecture: Raw Data → Predictions', fontsize=14, fontweight='bold', pad=15)

# Box drawing helper
def draw_box(ax, x, y, w, h, text, color, fontsize=10):
    rect = plt.Rectangle((x - w/2, y - h/2), w, h,
                          facecolor=color, edgecolor='black', linewidth=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', zorder=3, wrap=True, multialignment='center')

def draw_arrow(ax, x1, x2, y):
    ax.annotate('', xy=(x2 - 0.05, y), xytext=(x1 + 0.05, y),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Input data boxes
draw_box(ax, 1.0, 3.5, 1.5, 0.8, 'Numeric\nColumns\n(with NaN)', '#AED6F1', 9)
draw_box(ax, 1.0, 1.5, 1.5, 0.8, 'Categorical\nColumns\n(strings)', '#A9DFBF', 9)

# ColumnTransformer
draw_box(ax, 3.0, 3.5, 1.6, 0.8, 'Imputer\n(median)', '#AED6F1', 9)
draw_box(ax, 3.0, 1.5, 1.6, 0.8, 'Imputer\n(most_frequent)', '#A9DFBF', 9)
draw_box(ax, 5.0, 3.5, 1.6, 0.8, 'StandardScaler', '#AED6F1', 9)
draw_box(ax, 5.0, 1.5, 1.6, 0.8, 'OneHotEncoder', '#A9DFBF', 9)

# Combine
draw_box(ax, 7.0, 2.5, 1.5, 0.8, 'Combined\nFeature\nMatrix', '#F9E79F', 9)

# Model
draw_box(ax, 9.0, 2.5, 1.5, 0.8, 'Linear\nRegression\n(model)', '#F1948A', 9)

# Arrows
draw_arrow(ax, 1.75, 2.2, 3.5)
draw_arrow(ax, 1.75, 2.2, 1.5)
draw_arrow(ax, 3.8, 4.2, 3.5)
draw_arrow(ax, 3.8, 4.2, 1.5)
draw_arrow(ax, 5.8, 6.25, 3.5)
draw_arrow(ax, 5.8, 6.25, 1.5)
ax.annotate('', xy=(7.0, 2.9), xytext=(6.75, 3.1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(7.0, 2.1), xytext=(6.75, 1.9),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
draw_arrow(ax, 7.75, 8.25, 2.5)

# Label the ColumnTransformer bracket
ax.text(4.0, 4.6, 'ColumnTransformer', ha='center', fontsize=11,
        color='navy', fontweight='bold')
ax.text(5.0, 0.6, 'Different transforms applied to different column types',
        ha='center', fontsize=10, color='gray', style='italic')

plt.tight_layout()
plt.show()

## Section 5: Cross-Validation — More Reliable Model Comparison

### The Problem with a Single Train/Test Split

When we split data once into 80% train / 20% test, our results depend on **which 60 houses** ended up in the test set. If, by chance, the test set happens to have unusually expensive or cheap houses, our evaluation will be misleading.

### Cross-Validation: The Solution

**K-Fold Cross-Validation** solves this by repeating the evaluation K times:

```
Full dataset (300 houses) → shuffled and split into 5 'folds' of 60 houses each

Round 1: Train on folds [2,3,4,5] → Test on fold [1] → score_1
Round 2: Train on folds [1,3,4,5] → Test on fold [2] → score_2
Round 3: Train on folds [1,2,4,5] → Test on fold [3] → score_3
Round 4: Train on folds [1,2,3,5] → Test on fold [4] → score_4
Round 5: Train on folds [1,2,3,4] → Test on fold [5] → score_5

Final score = mean(score_1..5) ± std(score_1..5)
```

Every house appears in the test set exactly once. The mean score is a much more reliable estimate of true model performance.

Analogy: instead of taking one exam to judge a student, give them 5 different versions of the exam and average the results.

In [ ]:
# ── Cross-validation: compare 3 models fairly ───────────────────────────────
#
# We use Pipelines inside cross_val_score — this GUARANTEES no data leakage!
# The pipeline is re-fit from scratch on each training fold.

# Rebuild pipelines for the numeric-only feature set
pipeline_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression())
])

pipeline_dt = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   DecisionTreeRegressor(max_depth=5, random_state=42))
])

pipeline_knn = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   KNeighborsRegressor(n_neighbors=5, weights='distance'))
])

# Use the numeric-only X_num with NaN still present — pipeline handles imputation
X_cv = df[numeric_cols]   # raw, still has NaN
y_cv = df['price']

# cross_val_score: 5-fold CV, scoring by R²
# Returns array of 5 R² scores, one per fold
cv_scores = {}

for name, pipeline in [
    ('Linear Regression',   pipeline_lr),
    ('Decision Tree (d=5)', pipeline_dt),
    ('KNN (k=5)',           pipeline_knn),
]:
    scores = cross_val_score(
        pipeline,    # the pipeline to evaluate
        X_cv,        # full feature matrix (NOT pre-split)
        y_cv,        # full target vector
        cv=5,        # 5-fold cross-validation
        scoring='r2' # use R² as the metric
    )
    cv_scores[name] = scores
    print(f'{name}:')
    print(f'  Fold scores: {[f"{s:.3f}" for s in scores]}')
    print(f'  Mean R² = {scores.mean():.4f} ± {scores.std():.4f}')
    print()

print('The ± std tells you how STABLE the model is across different data subsets.')
print('Prefer models with HIGH mean AND LOW std.')

In [ ]:
# ── Visualize: Cross-validation comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Cross-Validation Model Comparison (5-Fold)', fontsize=14, fontweight='bold')

model_names  = list(cv_scores.keys())
means        = [cv_scores[m].mean() for m in model_names]
stds         = [cv_scores[m].std()  for m in model_names]
colors_bar   = ['steelblue', 'coral', 'mediumseagreen']

# Left plot: bar chart with error bars
bars = axes[0].bar(model_names, means, yerr=stds,
                   color=colors_bar, capsize=8, alpha=0.85,
                   edgecolor='black', linewidth=1)
axes[0].set_title('Mean R² ± Std (5-Fold CV)', fontsize=12)
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(0.7, color='red', linestyle='--', alpha=0.5, label='R²=0.7 threshold')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

# Add value labels on bars
for bar, mean, std in zip(bars, means, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                 f'{mean:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Right plot: strip chart showing all 5 fold scores per model
all_scores = [cv_scores[m] for m in model_names]

for i, (name, scores, color) in enumerate(zip(model_names, all_scores, colors_bar)):
    # Jitter x positions slightly for visibility
    x_positions = np.ones(5) * i + np.random.uniform(-0.15, 0.15, 5)
    axes[1].scatter(x_positions, scores, color=color, s=80, alpha=0.8,
                    edgecolor='black', zorder=3, label=name)
    # Draw mean line
    axes[1].hlines(scores.mean(), i - 0.25, i + 0.25, color=color,
                   linewidth=3, label=f'{name} mean')

axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels(model_names, rotation=15)
axes[1].set_title('Individual Fold Scores (5 dots per model)', fontsize=12)
axes[1].set_ylabel('R² Score per fold')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Also run CV with MAE scoring
print('Cross-validation with MAE scoring (negative MAE → lower is better):')
for name, pipeline in [('Linear Regression', pipeline_lr), ('Decision Tree', pipeline_dt)]:
    scores = cross_val_score(pipeline, X_cv, y_cv, cv=5, scoring='neg_mean_absolute_error')
    mae_scores = -scores   # flip sign: sklearn uses negative for 'higher is better' convention
    print(f'  {name}: MAE = ${mae_scores.mean():,.0f} ± ${mae_scores.std():,.0f}')

In [ ]:
# ── Full end-to-end pipeline with ColumnTransformer — the production pattern ──
#
# This is the COMPLETE, professional-grade sklearn workflow:
#   1. Define numeric & categorical column lists
#   2. Build sub-pipelines for each type
#   3. Combine with ColumnTransformer
#   4. Wrap with model in a Pipeline
#   5. Evaluate with cross-validation

# Column definitions
num_features  = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'condition_encoded']
cat_features  = ['neighborhood']

# Sub-pipelines
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

# ColumnTransformer
ct = ColumnTransformer([
    ('numeric',      num_pipe, num_features),
    ('categorical',  cat_pipe, cat_features)
])

# Three full pipelines — only the model step differs
prod_pipelines = {
    'LinearRegression':      Pipeline([('prep', ct), ('model', LinearRegression())]),
    'DecisionTree(d=5)':     Pipeline([('prep', ct), ('model', DecisionTreeRegressor(max_depth=5, random_state=42))]),
    'KNN(k=5)':              Pipeline([('prep', ct), ('model', KNeighborsRegressor(n_neighbors=5))]),
}

X_full = df[num_features + cat_features]   # raw data including NaN and strings
y_full = df['price']

print('Final Cross-Validation — Full Pipeline (numeric + categorical features):')
print('='*60)

results = []
for model_name, pipeline in prod_pipelines.items():
    scores = cross_val_score(pipeline, X_full, y_full, cv=5, scoring='r2')
    results.append({'Model': model_name, 'Mean R²': scores.mean(), 'Std R²': scores.std()})
    print(f'{model_name:30s}  R² = {scores.mean():.4f} ± {scores.std():.4f}')

best = max(results, key=lambda r: r['Mean R²'])
print(f'\nBest model: {best["Model"]}  (R² = {best["Mean R²"]:.4f})')
print('\nThis is the recommended pattern for any sklearn ML project:')
print('  raw DataFrame → ColumnTransformer → model → cross_val_score')

In [ ]:
# ── Feature importance visualization (Decision Tree) ─────────────────────────
# As a bonus: Decision Trees expose .feature_importances_ — how much each
# feature contributed to reducing prediction error in the tree splits.

# Fit the Decision Tree pipeline on all data for inspection
dt_full_pipe = prod_pipelines['DecisionTree(d=5)']
dt_full_pipe.fit(X_full, y_full)

# Get the fitted model from the pipeline
dt_fitted = dt_full_pipe.named_steps['model']

# Get feature names after ColumnTransformer transformation
ct_fitted = dt_full_pipe.named_steps['prep']
num_feature_names = num_features
cat_feature_names = ct_fitted.named_transformers_['categorical']['encoder'].get_feature_names_out(cat_features).tolist()
all_feature_names = num_feature_names + cat_feature_names

importances = dt_fitted.feature_importances_

# Sort by importance
feat_imp_df = pd.DataFrame({
    'Feature':    all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
colors_imp = ['#E74C3C' if imp == feat_imp_df['Importance'].max() else '#3498DB'
              for imp in feat_imp_df['Importance']]

bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
               color=colors_imp, edgecolor='white', alpha=0.85)

ax.set_title('Decision Tree: Feature Importances\n(Which features best predict house price?)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (fraction of variance reduction)')

# Add value labels
for bar, imp in zip(bars, feat_imp_df['Importance']):
    ax.text(imp + 0.001, bar.get_y() + bar.get_height()/2,
            f'{imp:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('The most important feature is:', feat_imp_df.iloc[-1]['Feature'])
print('This aligns with our data generation: price = 80 * size_sqft + ...')

## Summary: The Scikit-Learn API — Key Takeaways

### The 3-Method API

| Method | Who uses it | What it does |
|--------|-------------|---------------|
| `.fit(X, y)` | All estimators | Learn parameters from training data |
| `.transform(X)` | Preprocessors (scalers, encoders, imputers) | Apply the learned transformation |
| `.fit_transform(X)` | Preprocessors | Shortcut: fit + transform in one call |
| `.predict(X)` | Models (regression, classification) | Predict target values |
| `.score(X, y)` | All estimators | Evaluate performance (default: R² for regression) |

### The Golden Rules

1. **Never fit on test data.** Always call `.fit()` on training data only. Use `.transform()` on test data.

2. **Use Pipelines.** They prevent data leakage automatically and make code reproducible.

3. **Prefer cross-validation** over a single train/test split for model comparison.

4. **Scale before distance-based models.** KNN and SVMs are highly sensitive to feature scale; linear models benefit from it too.

5. **Use ColumnTransformer** for mixed-type data — apply different preprocessing to numeric vs categorical columns.

### The Full Workflow in One Picture

```
Raw DataFrame (NaN values, strings, different scales)
          │
          ▼
ColumnTransformer
 ├── Numeric: SimpleImputer → StandardScaler
 └── Categorical: SimpleImputer → OneHotEncoder
          │
          ▼
Model (LinearRegression / DecisionTree / KNN)
          │
          ▼
Predictions
          │
          ▼
cross_val_score → reliable estimate of real-world performance
```

## Self-Check Questions

Test your understanding. Try to answer before revealing the answers.

---

**Question 1:** Why should you fit the `StandardScaler` on TRAINING data only, and not on the full dataset?

<details>
<summary>Click to reveal answer</summary>

If you fit the scaler on the full dataset, the scaler's mean and standard deviation are computed using information from the TEST set. When you later evaluate the model on the test set, it has effectively already "seen" the test data through the scaling statistics. This is called **data leakage** — your evaluation results will be optimistically biased, and the model will perform worse in production than your metrics suggest. The correct workflow: fit on train, apply (transform) to both train and test.

</details>

---

**Question 2:** You have 1000 houses. You scale ALL features first (using the full 1000), then split 800/200 train/test. What is wrong with this order?

<details>
<summary>Click to reveal answer</summary>

This is data leakage. By scaling on all 1000 houses BEFORE splitting, the scaler's mean and standard deviation include information from the 200 test houses. The 200 test houses are no longer truly unseen — their statistics have influenced the preprocessing. You should always **split first, then fit preprocessing on training data only**. A Pipeline enforces this automatically: it fits all steps (including the scaler) only on the training fold.

</details>

---

**Question 3:** `pipeline.fit(X_train, y_train)` — what does this call do internally for a pipeline with `[StandardScaler, LinearRegression]`?

<details>
<summary>Click to reveal answer</summary>

The pipeline executes these steps in order:
1. `StandardScaler.fit_transform(X_train)` — computes mean and std from `X_train`, then returns the scaled version
2. `LinearRegression.fit(X_train_scaled, y_train)` — learns coefficients from the scaled training data

Later, when you call `pipeline.predict(X_test)`, it does:
1. `StandardScaler.transform(X_test)` — applies the SAME mean/std learned from training (no re-fitting!)
2. `LinearRegression.predict(X_test_scaled)` — returns predicted prices

</details>

---

**Question 4:** Your R² is 0.85 on training data but 0.42 on test data. What problem does this indicate, and what might you do about it?

<details>
<summary>Click to reveal answer</summary>

This large gap between training R² (0.85) and test R² (0.42) indicates **overfitting**. The model has memorised patterns specific to the training data (including noise) rather than learning generalizable relationships. It performs well on data it has seen but poorly on new data.

Potential fixes:
- **Reduce model complexity**: limit `max_depth` for Decision Trees, reduce `n_estimators`, add regularization (Ridge, Lasso instead of LinearRegression)
- **Get more training data**: more samples make it harder to memorise
- **Feature selection**: remove noisy or irrelevant features
- **Cross-validation**: use CV during model selection to catch this early

</details>

---

**Question 5:** What is the advantage of using a `Pipeline` over manually applying each preprocessing step?

<details>
<summary>Click to reveal answer</summary>

Pipelines offer several critical advantages:

1. **No data leakage**: The pipeline fits all preprocessing steps only on training data, automatically. If you use `cross_val_score(pipeline, ...)`, each fold correctly fits preprocessing on the training portion only.

2. **Reproducibility**: One `pipeline.fit()` call trains the entire chain. You can't accidentally forget to re-fit a scaler.

3. **Production safety**: `pipeline.predict(raw_data)` accepts raw data and applies all transformations automatically. No manual preprocessing needed at serving time.

4. **Cleaner code**: Instead of 6+ manual steps, you have one `Pipeline` definition and one `.fit()` call.

5. **Hyperparameter tuning**: `GridSearchCV` can tune hyperparameters of any step in the pipeline (e.g., `model__max_depth=5`) because it can access all steps by name.

</details>

In [ ]:
# ── Quick Reference: The sklearn API Cheat Sheet ──────────────────────────────
print('='*65)
print('SCIKIT-LEARN API QUICK REFERENCE')
print('='*65)

cheat_sheet = [
    ('PREPROCESSING', ''),
    ('StandardScaler()',                'subtract mean, divide by std → mean=0, std=1'),
    ('MinMaxScaler()',                  'scale to [0, 1] range'),
    ('SimpleImputer(strategy=...)',     'fill NaN: mean/median/most_frequent'),
    ('OneHotEncoder(drop="first")',     'nominal categories → binary dummy columns'),
    ('LabelEncoder()',                  'ordinal categories → integer codes'),
    ('', ''),
    ('MODELS', ''),
    ('LinearRegression()',              'finds best linear fit (weights per feature)'),
    ('DecisionTreeRegressor()',         'if/else rule tree on features'),
    ('KNeighborsRegressor(n_neighbors=5)', 'avg price of K most similar houses'),
    ('', ''),
    ('MODEL SELECTION', ''),
    ('train_test_split(X, y, test_size=0.2)', 'split into train and test sets'),
    ('cross_val_score(pipe, X, y, cv=5)', 'k-fold cross-validation scores'),
    ('', ''),
    ('METRICS', ''),
    ('r2_score(y_true, y_pred)',         'fraction of variance explained [0,1]'),
    ('mean_absolute_error(y, y_pred)',   'average |actual - predicted|'),
    ('mean_squared_error(y, y_pred)',    'average (actual - predicted)²'),
    ('', ''),
    ('PIPELINE', ''),
    ('Pipeline([("name", step), ...])',  'chain preprocessing + model'),
    ('ColumnTransformer([("n", t, cols)])','different transforms per column group'),
]

for item, desc in cheat_sheet:
    if desc == '' and item != '':
        print(f'\n  [{item}]')
    elif item == '':
        pass
    else:
        print(f'  {item:<45} {desc}')

print('\n' + '='*65)
print('Remember the Golden Rule:')
print('  .fit() on TRAINING data only. .transform()/.predict() on any data.')
print('  Use a Pipeline — it enforces this rule automatically.')
print('='*65)